# Matrix Alpha // Firestarter SPB
## Binance Top 100 Dataset Profiling Viewer

> **[IMPORTANT] WARNINGS & CONSTRAINTS**
> - **Research Viewer Only:** This notebook is strictly for offline inspection, profiling, and quality validation of the ingested dataset.
> - **No Cell 2 / Labels:** This notebook does not generate target labels, features, or predictive variables.
> - **No Model Training:** There is no machine learning model training or parameter optimization here.
> - **No Trading Recommendations / Strategy Claims:** This notebook does not contain any trading strategies, execution logic, or financial recommendations.
> - **No ER / FMLC / Flowprint Computation:** No custom volatility or flow-based indicator calculation is performed in this version.

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# Set styling for premium visual feel
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

# Dataset Path
DATA_DIR = "../data/research/binance_top100_excluding_existing_5_1month"
print(f"Dataset target path configured: {DATA_DIR}")

### Symbol Selection

**Instruction:** 
To analyze a different symbol:
1. Update the `SELECTED_SYMBOL` variable in the cell below to any valid symbol from the inventory list (e.g., `"BTCUSDT"`, `"ETHUSDT"`, or `"币安人生USDT"`).
2. Run the notebook from that cell down.

In [ ]:
# UPDATE THIS VARIABLE TO INSPECT A SPECIFIC SYMBOL
SELECTED_SYMBOL = "BTCUSDT"

print(f"Selected Symbol for detailed analysis: {SELECTED_SYMBOL}")

### 1. Dataset Inventory Audit
The cell below scans the directory and aggregates metadata for all 100 symbols, generating a summary audit table showing:
- Symbol name
- Total row count
- First and last timestamp (both raw and human-readable UTC)
- Number of missing 5-minute candles
- Non-standard symbol flag (indicating Unicode characters)

In [ ]:
def build_inventory(data_dir):
    csv_files = glob.glob(os.path.join(data_dir, "*_1month_5m.csv"))
    inventory = []
    
    for filepath in csv_files:
        filename = os.path.basename(filepath)
        symbol = filename.replace("_1month_5m.csv", "")
        
        try:
            # Read first and last row to check timestamps and count
            df_temp = pd.read_csv(filepath)
            row_count = len(df_temp)
            
            if row_count > 0:
                first_ts = int(df_temp.iloc[0]['open_time'])
                last_ts = int(df_temp.iloc[-1]['open_time'])
                
                # Expected count based on time span (5m steps)
                expected_span_count = int((last_ts - first_ts) / 300000) + 1
                missing_5m = max(0, expected_span_count - row_count)
                
                first_dt = pd.to_datetime(first_ts, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M:%S')
                last_dt = pd.to_datetime(last_ts, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M:%S')
            else:
                first_ts, last_ts = np.nan, np.nan
                first_dt, last_dt = "N/A", "N/A"
                missing_5m = 0
                
            # Check for non-standard/Unicode characters in symbol
            nonstandard_flag = not symbol.isascii()
            
            inventory.append({
                "symbol": symbol,
                "row_count": row_count,
                "first_timestamp": first_ts,
                "last_timestamp": last_ts,
                "first_time_utc": first_dt,
                "last_time_utc": last_dt,
                "missing_5m_count": missing_5m,
                "nonstandard_symbol_flag": nonstandard_flag
            })
        except Exception as e:
            print(f"Error reading {symbol}: {e}")
            
    df_inv = pd.DataFrame(inventory)
    if not df_inv.empty:
        df_inv = df_inv.sort_values(by="symbol")
    return df_inv

df_inventory = build_inventory(DATA_DIR)
print(f"Loaded metadata for {len(df_inventory)} symbols.")
df_inventory

### 2. Selected Symbol Detailed Profiling

We now load the full 1-month 5m dataset for `SELECTED_SYMBOL` and inspect:
- First/last 20 rows of the dataset.
- Exact numeric summary tables.
- Volume & Volatility statistics.

In [ ]:
csv_path = os.path.join(DATA_DIR, f"{SELECTED_SYMBOL}_1month_5m.csv")
if not os.path.exists(csv_path):
    # Try finding via pattern in case of encoding mismatches
    files = glob.glob(os.path.join(DATA_DIR, f"*{SELECTED_SYMBOL}*_5m.csv"))
    if files:
        csv_path = files[0]
        print(f"Resolved path for non-standard symbol: {csv_path}")
    else:
        raise FileNotFoundError(f"No CSV file found for symbol: {SELECTED_SYMBOL}")

# Load raw CSV
df = pd.read_csv(csv_path)

# Convert timestamps to datetime objects
df['open_datetime'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
df['close_datetime'] = pd.to_datetime(df['close_time'], unit='ms', utc=True)

print(f"Successfully loaded {len(df)} rows for {SELECTED_SYMBOL}")
# Preview latest 20 rows
df.tail(20)

In [ ]:
# 1. Basic OHLCV Stats
print("--- OHLCV Summary Statistics ---")
print(df[['open', 'high', 'low', 'close', 'volume']].describe())

# 2. Volume Details
print("\n--- Volume Analysis ---")
vol_stats = {
    "Total Volume (Base)": df['volume'].sum(),
    "Average Volume/Candle": df['volume'].mean(),
    "Max Volume/Candle": df['volume'].max(),
    "Std Volume": df['volume'].std()
}
for k, v in vol_stats.items():
    print(f"{k}: {v:,.4f}")

# 3. Volatility/Range Analysis
# We define Range as (High - Low) and Range % as (High - Low) / Open * 100
df['candle_range'] = df['high'] - df['low']
df['range_pct'] = (df['candle_range'] / df['open']) * 100

print("\n--- Volatility & Candle Range Metrics ---")
print(df[['candle_range', 'range_pct']].describe())

### 3. Multi-Scale Timeframe Resampling
We resample the 5-minute ticks to 1-hour and 4-hour candles. OHLC values are extracted using standard open/high/low/close rules, and volumes are aggregated.

In [ ]:
# Ensure index is datetime for resampling
df_resample = df.set_index('open_datetime')

# 1-Hour Aggregation
df_1h = df_resample.resample('1h').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'quote_asset_volume': 'sum',
    'trades': 'sum'
}).dropna()

# 4-Hour Aggregation
df_4h = df_resample.resample('4h').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'quote_asset_volume': 'sum',
    'trades': 'sum'
}).dropna()

print(f"Resampled 1-Hour Shape: {df_1h.shape}")
print(f"Resampled 4-Hour Shape: {df_4h.shape}")

### 4. Chart Visualizations
We generate three panels:
1. **Price Line Chart** (1-Hour Close Price)
2. **Volume Bar Chart** (1-Hour Volume)
3. **Volatility/Candle Range Panel** (1-Hour High-Low Volatility % or Range)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Close Price
axes[0].plot(df_1h.index, df_1h['close'], color='#0066CC', linewidth=1.5, label='1h Close Price')
axes[0].set_title(f"{SELECTED_SYMBOL} - 1h Close Price", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Price (USDT)")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# Panel 2: Volume
axes[1].bar(df_1h.index, df_1h['volume'], color='#009966', width=0.03, alpha=0.8, label='1h Volume')
axes[1].set_title(f"{SELECTED_SYMBOL} - 1h Trading Volume", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Volume")
axes[1].legend(loc="upper left")
axes[1].grid(True, alpha=0.3)

# Panel 3: Volatility (Range %)
df_1h['range_pct'] = ((df_1h['high'] - df_1h['low']) / df_1h['open']) * 100
axes[2].plot(df_1h.index, df_1h['range_pct'], color='#CC3333', linewidth=1.2, alpha=0.9, label='1h High-Low Range %')
axes[2].set_title(f"{SELECTED_SYMBOL} - Volatility (High-Low % of Open)", fontsize=12, fontweight='bold')
axes[2].set_ylabel("Volatility %")
axes[2].set_xlabel("Time (UTC)")
axes[2].legend(loc="upper left")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()